# Explore how the collect() operation works in PySpark using a dataset with basic RDD operations.
---
- Explore how the **collect() operation** works in PySpark.  
- Use a dataset to demonstrate **basic RDD operations**.  
- Data is loaded and parallelized into an RDD.  
- Apply transformations like `map()`, `filter()`, and `sortBy()`.  
- Use `collect()` to trigger execution and retrieve results to the driver.  
- Helps in understanding the difference between **transformations (lazy)** and **actions (execution)** in Spark.  

## Dataset Overview

- **Total Records:** 50 students  
- **Columns (7):** `id`, `name`, `age`, `gender`, `math`, `science`, `english`  
- **Data Quality:** No missing values  

---

###  Demographics
- **Age Range:** 18 – 25 years  
- **Average Age:** ≈ 21.5 years  
- **Gender Distribution:** 29 Female, 21 Male  

---

###  Academic Performance

###  Math
- **Range:** 40 – 100  
- **Mean:** 68.9  
- **Standard Deviation:** 17.6 (high variation)  

###  Science
- **Range:** 44 – 99  
- **Mean:** 70.2  
- **Standard Deviation:** 14.6 (moderate variation)  

###  English
- **Range:** 42 – 100  
- **Mean:** 69.4
- **Standard Deviation:** 18.7 (highest variation)  

---

###  Key Insights
- Science is the strongest subject on average.  
- English shows the highest variation in performance.  
- Student performance varies significantly across subjects (not uniform).  

In [2]:
sc

<SparkContext master=local[*] appName=PySparkShell>

In [4]:
# from pyspark import SparkContext

# # Initialize SparkContext
# sc = SparkContext("local", "CSV_RDD_Example")

In [5]:
# Load CSV file (assuming students.csv is in working directory)
data = sc.textFile("students.csv")

In [6]:
# Step 1: Remove header
header = data.first()
rows = data.filter(lambda line: line != header)

In [7]:
# Step 2: Split by comma
split_rdd = rows.map(lambda line: line.split(","))

In [8]:
print("=== Student Dataset (first 10 rows) ===")
for row in split_rdd.take(10):   # you can change 10 → 20, 50 etc.
    print(row)

=== Student Dataset (first 10 rows) ===
['1', 'Alice', '20', 'F', '66', '92', '44']
['2', 'Bob', '20', 'M', '82', '52', '77']
['3', 'Charlie', '22', 'F', '43', '57', '76']
['4', 'David', '19', 'M', '95', '69', '46']
['5', 'Eva', '19', 'F', '62', '44', '96']
['6', 'Frank', '22', 'F', '70', '78', '94']
['7', 'Grace', '24', 'F', '67', '66', '93']
['8', 'Henry', '21', 'F', '53', '82', '60']
['9', 'Ivy', '19', 'M', '64', '52', '46']
['10', 'Jack', '19', 'F', '44', '59', '60']


In [10]:
# Step 3: Convert fields into structured format
# (id, name, age, gender, math, science, english)
students_rdd = split_rdd.map(lambda x: (int(x[0]), x[1], int(x[2]), x[3], int(x[4]), int(x[5]), int(x[6])))

In [11]:
# Step 4: Calculate average marks for each student
avg_marks_rdd = students_rdd.map(lambda x: (x[1], (x[4] + x[5] + x[6]) / 3))

In [12]:
# Step 5: Filter students who scored avg >= 75
passed_rdd = avg_marks_rdd.filter(lambda x: x[1] >= 75)

In [13]:
# Step 6: Sort students by avg marks (descending)
sorted_passed_rdd = passed_rdd.sortBy(lambda x: x[1], ascending=False)

In [14]:
# Step 7: Collect results to driver
results = sorted_passed_rdd.collect()

In [15]:
# Print results
print("=== Students with Average >= 75 ===")
for student in results:
    print(f"Name: {student[0]}, Avg Marks: {student[1]:.2f}")

=== Students with Average >= 75 ===
Name: Leo, Avg Marks: 88.00
Name: Olivia, Avg Marks: 88.00
Name: Rita, Avg Marks: 86.67
Name: Kathy, Avg Marks: 81.67
Name: George, Avg Marks: 81.67
Name: Frank, Avg Marks: 80.67
Name: Oscar, Avg Marks: 80.00
Name: Uma, Avg Marks: 78.33
Name: Kyle, Avg Marks: 78.33
Name: Matt, Avg Marks: 78.33
Name: Tina, Avg Marks: 76.00
Name: Victor, Avg Marks: 75.67
Name: Grace, Avg Marks: 75.33
Name: Mona, Avg Marks: 75.00
Name: Will, Avg Marks: 75.00


In [17]:
# Step 8: Some extra RDD operations for practice

In [18]:
# (a) Count how many students passed
count_passed = passed_rdd.count()
print("\nNumber of students who passed:", count_passed)


Number of students who passed: 15


In [19]:
# (a) Count how many students passed
count_passed = passed_rdd.count()
print("\nNumber of students who passed:", count_passed)


Number of students who passed: 15


In [20]:
# (c) Show first 5 passed students
print("\nFirst 5 Passed Students (via take):")
print(passed_rdd.take(5))


First 5 Passed Students (via take):
[('Frank', 80.66666666666667), ('Grace', 75.33333333333333), ('Kathy', 81.66666666666667), ('Leo', 88.0), ('Mona', 75.0)]


# Summary

This experiment demonstrates **data processing and analysis using PySpark RDDs** on a student dataset.  
The dataset contains 50 student records with details such as ID, name, age, gender, and marks in three subjects (Math, Science, English).  

---

##  Operations Performed

1. **Data Loading**
   - Loaded `students.csv` using `sc.textFile()`.  
   - Removed header row and split records by comma.  

2. **Data Transformation**
   - Converted raw rows into structured tuples: `(id, name, age, gender, math, science, english)`.  
   - Calculated average marks per student using `map()`.  

3. **Filtering**
   - Selected students with average marks ≥ 75.  

4. **Sorting**
   - Sorted qualified students by average marks in descending order using `sortBy()`.  

5. **Actions**
   - `collect()` → Retrieved and displayed passed students.  
   - `count()` → Counted number of students who passed (15 students).  
   - `reduce()` → Found the topper (highest average).  
   - `take(n)` → Displayed first 5 passed students.  

---

##  Results

- **Students Passed (Avg ≥ 75):** 15  
- **Topper:** Olivia (Avg = 88.0)  
- **Highest Scorers:** Leo (88.0), Olivia (88.0), Rita (86.7)  
- **Example Output:**  
